In [ ]:
from pathlib import Path
import cv2
import numpy as np



In [ ]:
# Validate all image/mask pairs before writing YOLO polygon labels.
ROOT = Path.cwd()
if not (ROOT / "new_dataset.ipynb").is_file():
    raise RuntimeError("Run the notebook from its project root (the notebook directory).")
dataset_dir = ROOT / "optic_disk_kfold_reviewed_dataset"
images_dir = dataset_dir / "images"
masks_dir = dataset_dir / "masks"
labels_dir = dataset_dir / "labels"
CLASS_ID = 0

for folder in (images_dir, masks_dir):
    if not folder.is_dir():
        raise FileNotFoundError(f"Missing directory: {folder}")

image_paths = sorted(images_dir.glob("*.png"))
mask_paths = sorted(masks_dir.glob("*.png"))
if not image_paths or not mask_paths:
    raise ValueError("Images and masks directories must contain PNG files.")

expected_masks = {f"{image.stem}_mask.png" for image in image_paths}
actual_masks = {mask.name for mask in mask_paths}
if expected_masks != actual_masks:
    raise ValueError(
        f"Missing masks: {sorted(expected_masks - actual_masks)}; "
        f"Masks without an image: {sorted(actual_masks - expected_masks)}"
    )

pending_labels = []
errors = []
for image_path in image_paths:
    mask_path = masks_dir / f"{image_path.stem}_mask.png"
    image = cv2.imread(str(image_path), cv2.IMREAD_UNCHANGED)
    mask = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)
    if image is None or mask is None:
        errors.append(f"{image_path.name}: unreadable image or mask")
        continue
    # RGB grayscale exports are valid only when all three channels agree.
    # Extracting one identical channel preserves every pixel value exactly.
    if mask.ndim == 3 and mask.shape[2] == 3:
        if (np.array_equal(mask[:, :, 0], mask[:, :, 1])
                and np.array_equal(mask[:, :, 0], mask[:, :, 2])):
            mask = mask[:, :, 0]
    if mask.ndim != 2:
        errors.append(f"{mask_path.name}: expected a grayscale mask (2D or RGB with identical channels)")
        continue
    if image.shape[:2] != mask.shape:
        errors.append(f"{mask_path.name}: image and mask dimensions differ")
        continue

    # PNG masks may contain gray edge pixels from interpolation/antialiasing.
    # Preserve binary 0/1 masks; threshold 8-bit grayscale masks at 127.
    if mask.dtype != np.uint8:
        errors.append(f"{mask_path.name}: expected an 8-bit grayscale mask")
        continue
    threshold = 0 if mask.max() <= 1 else 127
    mask_bin = (mask > threshold).astype(np.uint8) * 255
    contours, hierarchy = cv2.findContours(
        mask_bin, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE
    )
    if len(contours) != 1:
        errors.append(
            f"{mask_path.name}: expected one filled disc without holes, "
            f"found {len(contours)} contours; review the mask"
        )
        continue
    contour = contours[0]
    if len(contour) < 3 or cv2.contourArea(contour) <= 0:
        errors.append(f"{mask_path.name}: degenerate polygon")
        continue

    h, w = mask.shape
    normalized = contour.reshape(-1, 2).astype(np.float64)
    normalized[:, 0] /= w
    normalized[:, 1] /= h
    coordinates = " ".join(f"{value:.6f}" for value in normalized.flatten())
    pending_labels.append((image_path.stem + ".txt", f"{CLASS_ID} {coordinates}\n"))

if errors:
    raise ValueError("No labels written. Correct these masks first:\n" + "\n".join(errors))

# Refuse to mix the conversion with existing labels from another run.
if labels_dir.exists() and any(labels_dir.iterdir()):
    raise FileExistsError("The labels directory must be empty before conversion.")
labels_dir.mkdir(parents=True, exist_ok=True)
for filename, content in pending_labels:
    (labels_dir / filename).write_text(content, encoding="utf-8")

print(f"Conversion completed: {len(pending_labels)} labels written to {labels_dir}")


## Validation croisée par patient — dataset reviewed

Même organisation que l'ancien notebook : placer `kfold_train_reviewed.csv` et le dossier `optic_disk_kfold_reviewed_dataset` à côté de ce notebook, puis l'ouvrir avec ce dossier comme répertoire de travail (racine du projet).

Les scripts Python filtrent les vues `done` et copient les images/masques ; **cette cellule crée les cinq folds**, sans utiliser le manifeste d'une ancienne exécution du builder. Les vues d'un même patient restent ensemble. Priorité aux effectifs images/patients, puis recherche reproductible d'un équilibre des `PATHO`, `ANNOTATION_STATUS` et côtés parmi 2 000 répartitions candidates (graine 42). Le statut d'annotation est un indicateur approximatif, pas une mesure de difficulté. Une pathologie vide signifie non renseignée. Pour 38 paires et 4 vues seules : 16 images et 8 ou 9 patients par fold. Ne pas choisir la graine en fonction des performances du modèle.

Les sorties sont `folds/fold_N/train.txt` et `val.txt`. Chaque ligne est relative à la racine du projet, par exemple `optic_disk_kfold_reviewed_dataset/images/0000.png`, comme auparavant. Les futurs YAML doivent utiliser cette même racine. Aucun YAML ni entraînement n'est généré.

Une réexécution conserve la même répartition. Les listes absolues issues de la version précédente peuvent être remplacées par les chemins relatifs si les images sont identiques. Toute partition existante différente est refusée avant écriture : choisir alors un autre dossier `OUTPUT`. Les anciens CSV de folds sont contrôlés s'ils existent, mais ne déterminent pas la nouvelle répartition.


In [ ]:
# Create folds here, as in the original notebook. TXT paths are repo-root-relative.
from pathlib import Path
import csv
import random
from collections import Counter, defaultdict

ROOT = Path.cwd()
CSV_PATH = ROOT / "kfold_train_reviewed.csv"
DATASET = ROOT / "optic_disk_kfold_reviewed_dataset"
OUTPUT = DATASET / "folds"
N_FOLDS = 5
SEED = 42
TRIALS = 2000

if not (ROOT / "new_dataset.ipynb").is_file():
    raise RuntimeError("Run the notebook from its project root (the notebook directory).")
if N_FOLDS < 2 or TRIALS < 1:
    raise ValueError("Require at least two folds and one candidate partition.")
with CSV_PATH.open(encoding="utf-8-sig", newline="") as stream:
    reader = csv.DictReader(stream)
    fields = reader.fieldnames or []
    if not {"PATIENT", "NAME", "SIDE", "Matthis CHECK"}.issubset(fields):
        raise ValueError("CSV requires PATIENT, NAME, SIDE and Matthis CHECK.")
    if len(fields) != len(set(fields)):
        raise ValueError("Duplicate CSV column names.")
    rows = list(reader)
if not rows:
    raise ValueError("CSV contains no images.")

groups = defaultdict(list)
seen_names, seen_eyes = set(), set()
for line, row in enumerate(rows, 2):
    if None in row or any(value is None for value in row.values()):
        raise ValueError(f"Line {line}: malformed CSV row.")
    patient, side = row["PATIENT"].strip().upper(), row["SIDE"].strip().upper()
    name = row["NAME"].strip()
    if not patient or side not in {"L", "R"}:
        raise ValueError(f"Line {line}: invalid patient or eye side.")
    if row["Matthis CHECK"].strip().lower() != "done":
        raise ValueError(f"Line {line}: only reviewed done images are allowed.")
    if not name or name in {".", ".."} or any(c in name for c in ("/", chr(92), ":")):
        raise ValueError(f"Line {line}: invalid image filename.")
    if name.casefold() in seen_names or (patient, side) in seen_eyes:
        raise ValueError(f"Line {line}: duplicate image or patient eye.")
    image = DATASET / "images" / name
    label = DATASET / "labels" / f"{image.stem}.txt"
    mask = DATASET / "masks" / f"{image.stem}_mask.png"
    for path in (image, label, mask):
        if not path.is_file():
            raise FileNotFoundError(path)
    if not label.read_text(encoding="utf-8-sig").strip():
        raise ValueError(f"Empty label: {label.name}")
    seen_names.add(name.casefold())
    seen_eyes.add((patient, side))
    groups[patient].append(row)
if len(groups) < N_FOLDS:
    raise ValueError("Fewer patients than folds.")
all_names = {row["NAME"].strip() for row in rows}
actual_images = {p.name for p in (DATASET / "images").iterdir() if p.is_file()}
if actual_images != all_names:
    raise ValueError("The images directory and CSV must contain exactly the same images.")

def features(rows):
    # Count per image, independently for each available metadata column.
    return Counter((column, row.get(column, '').strip().lower() or '<unspecified>')
                   for row in rows for column in ('PATHO', 'ANNOTATION_STATUS', 'SIDE'))


def assign_folds(groups, n_folds, seed, trials):
    by_size = {size: sorted(p for p, rows in groups.items() if len(rows) == size)
               for size in (1, 2)}
    slots = [{1: 0, 2: 0} for _ in range(n_folds)]
    for size in (2, 1):
        for _ in by_size[size]:
            fold = min(range(n_folds), key=lambda f: (
                2 * slots[f][2] + slots[f][1], slots[f][2] + slots[f][1], f))
            slots[fold][size] += 1
    vectors = {p: features(rows) for p, rows in groups.items()}
    totals = features([row for rows in groups.values() for row in rows])
    image_total = sum(map(len, groups.values()))
    rng = random.Random(seed)
    best, best_score = None, float('inf')
    for _ in range(trials):
        candidate = [[] for _ in range(n_folds)]
        for size in (2, 1):
            patients = by_size[size].copy()
            rng.shuffle(patients)
            offset = 0
            for f in range(n_folds):
                count = slots[f][size]
                candidate[f].extend(patients[offset:offset + count])
                offset += count
        score = 0.0
        for patients in candidate:
            observed = Counter()
            for patient in patients:
                observed.update(vectors[patient])
            image_count = sum(len(groups[p]) for p in patients)
            for key, total in totals.items():
                expected = total * image_count / image_total
                score += (observed[key] - expected) ** 2 / max(expected, 1.0)
        if score < best_score:
            best, best_score = candidate, score
    return {patient: f + 1 for f, patients in enumerate(best) for patient in patients}, best_score


assignments, stratification_score = assign_folds(groups, N_FOLDS, SEED, TRIALS)
all_patients = set(groups)
validation_seen = set()
pending = []
for fold in range(1, N_FOLDS + 1):
    val_patients = {p for p, assigned in assignments.items() if assigned == fold}
    train_patients = all_patients - val_patients
    train_names = {r["NAME"].strip() for p in train_patients for r in groups[p]}
    val_names = {r["NAME"].strip() for p in val_patients for r in groups[p]}
    if not val_names or train_patients & val_patients:
        raise RuntimeError("Empty validation fold or patient leakage.")
    if train_names & val_names or train_names | val_names != all_names:
        raise RuntimeError("Invalid image partition.")
    if validation_seen & val_names:
        raise RuntimeError("An image occurs in multiple validation folds.")
    validation_seen.update(val_names)
    folder = OUTPUT / f"fold_{fold}"
    for split, names, csv_name in (("train", train_names, "train.csv"),
                                    ("val", val_names, "valid.csv")):
        csv_path = folder / csv_name
        if csv_path.exists():
            with csv_path.open(encoding="utf-8-sig", newline="") as stream:
                existing = list(csv.DictReader(stream))
            if len(existing) != len(names) or {r.get("NAME", "").strip() for r in existing} != names:
                raise FileExistsError(f"Existing CSV partition differs: {csv_path}. Choose another OUTPUT.")
        relative_paths = sorted((DATASET / "images" / name).relative_to(ROOT).as_posix()
                                for name in names)
        content = "".join(path + "\n" for path in relative_paths)
        target = folder / f"{split}.txt"
        if target.exists():
            old_paths = target.read_text(encoding="utf-8").splitlines()
            absolute_paths = sorted((DATASET / "images" / name).resolve().as_posix() for name in names)
            normalized = sorted(path.replace(chr(92), "/") for path in old_paths)
            if normalized != relative_paths and normalized != absolute_paths:
                raise FileExistsError(f"Existing TXT partition differs: {target}. Choose another OUTPUT.")
        pending.append((target, content))
    print(f"Fold {fold}: train {len(train_names)} images / {len(train_patients)} patients; "
          f"val {len(val_names)} images / {len(val_patients)} patients")
if validation_seen != all_names:
    raise RuntimeError("Some images were never assigned to validation.")
# All partitions and existing outputs are checked before any writes.
for target, content in pending:
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding="utf-8")
print(f"Created/verified {len(pending)} TXT lists. Seed: {SEED}; trials: {TRIALS}.")
print(f"Paths are relative to project root: {ROOT}. Metadata balance score: {stratification_score:.3f}")
